In [1]:
import torch
import time
from challenge.tinyyolov2 import TinyYoloV2_pruned, TinyYoloV2_BNOpt

In [2]:
def benchmark(model, device, n=100, fp16=False):
    with torch.no_grad():
        input = torch.rand(1,3,320,320).to(device)
        
        if fp16:
            input = input.half()
            model.half()
        
        # initial run can vary a bit, so don't use it in the benchmark
        output = model(input)
        
        start = time.time()
        
        for _ in range(n):
            _ = model(input)
    
        end = time.time()
    
        return (end - start) * 1000 / n # average time in ms

In [3]:
#model_path = "../models/voc_pretrained_bnopt.pt"
#model_path = "../models/person_only_both_datasets_4_layers_finetuned/model_best.pt"
model_path = "../models/person_only_both_datasets_4_layers_finetuned/model_best_bnopt.pt"
#model_path = "../models/person_only_both_datasets_4_layers_finetuned/iterative_pruning_15_epochs_onestep/model_pruned_8.pt"

In [4]:
sd = torch.load(model_path)

device = torch.device('cuda')

#model = TinyYoloV2_pruned(num_classes=1)
model = TinyYoloV2_BNOpt(num_classes=1)
model.load_state_dict(sd)

model.to(device)
model.eval()


TinyYoloV2_BNOpt(
  (pad): ReflectionPad2d((0, 1, 0, 1))
  (conv1): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv4): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv5): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv6): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv7): Conv2d(512, 1024, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv8): Conv2d(1024, 1024, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv9): Conv2d(1024, 30, kernel_size=(1, 1), stride=(1, 1))
)

In [10]:
total_time = benchmark(model, device, fp16=True)
print(f'total time = {total_time:.1f}ms')

total time = 57.7ms
